# Repository inventory

A live manifest of what this repository tracks: the data files under `data/`
and the exported figures under `figures/`, both listed straight from disk so
the page always matches what is committed. Filenames carry processing
date-stamps that change on reprocessing, so this page is generated, not
hand-maintained. Re-run it after regenerating data or figures to refresh it.

The raw `00-*` inputs are tracked for provenance and never written to; the
derived stages (`01-` onward) are regenerated by the
[pipeline notebooks](../index.md) and tracked as a committed
snapshot. See [Data](../docs/data.md) for what each stage means and how it
maps onto the [processing versions](../docs/time-lag.qmd).

## Imports and helpers

In [1]:
import base64
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown, HTML

NB_START = datetime.now()  # notebook start time (reported in the last cell)

# Tracked datasets live under data/ (raw 00-* inputs and derived stages);
# exported figures live under figures/.
DATA = Path("../data")
FIGDIR = Path("../figures")


def human_size(n):
    """Human-readable byte size."""
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024


def scenario_of(name):
    """Leading scenario code in a filename (e.g. 'QCL-1'), or '' if none."""
    m = re.match(r"([A-Z]+-\d+)", name)
    return m.group(1) if m else ""


def folder_table(folder, pattern="*"):
    """One row per file in `folder` matching `pattern`: name, scenario, size."""
    rows = [
        {"file": f.name, "scenario": scenario_of(f.name),
         "size": human_size(f.stat().st_size)}
        for f in sorted(Path(folder).glob(pattern)) if f.is_file()
    ]
    return pd.DataFrame(rows)

## EddyPro flux output (`00-eddypro_fluxes_level-1/`)

One FLUXNET-format CSV per scenario, all five (`*-1` to `*-5`). For the PWB
scenario (`*-5`) the lag was removed from the raw data before the EddyPro run, so
that run applied no lag compensation. These are the raw inputs to
[notebook 01](01_read_fluxes_to_parquet.ipynb).

**Table.** Raw EddyPro flux output in `data/00-eddypro_fluxes_level-1/`: one
FLUXNET-format CSV per processing version, listed from disk with its size.

In [2]:
folder_table(DATA / "00-eddypro_fluxes_level-1", "*_adv.csv")

,file,scenario,size
0,LGR-1_eddypro_LGR-1_FR-20260619-164852_fluxnet...,LGR-1,27.9 MB
1,LGR-2_eddypro_LGR-2_FR-20260615-140834_fluxnet...,LGR-2,27.8 MB
2,LGR-3_eddypro_LGR-3_FR-20260615-140917_fluxnet...,LGR-3,27.8 MB
3,LGR-4_eddypro_LGR-4_FR-20260623-133726_fluxnet...,LGR-4,27.8 MB
4,LGR-5_eddypro_LGR-5_FR-20260622-174418_fluxnet...,LGR-5,27.8 MB
5,QCL-1_eddypro_QCL-1_FR-20260619-164952_fluxnet...,QCL-1,34.1 MB
6,QCL-2_eddypro_QCL-2_FR-20260615-140229_fluxnet...,QCL-2,33.9 MB
7,QCL-3_eddypro_QCL-3_FR-20260615-140311_fluxnet...,QCL-3,34.0 MB
8,QCL-4_eddypro_QCL-4_FR-20260623-134019_fluxnet...,QCL-4,34.0 MB
9,QCL-5_eddypro_QCL-5_FR-20260622-174805_fluxnet...,QCL-5,33.9 MB


## EddyPro settings (`00-eddypro_settings/`)

The EddyPro project (`.eddypro`) and metadata (`.metadata`) files that produced
each flux output. Every run shares the same site, period, raw signal, and all
other EddyPro options; the **only** thing that differs is how the N₂O / CH₄ time
lag is handled. The table reads that difference straight from the files: the time
lag method (`tlag_meth` in the `.eddypro`) and the per-gas search window (nominal
lag and min to max, from the active gas column in the `.metadata`). Keep these in
sync with the [processing versions](../docs/time-lag.qmd).

**Table.** Time-lag settings read straight from each version's EddyPro project
and metadata files: the time-lag method and the per-gas search window. Every
other setting is identical across versions.

In [3]:
# Read the actual time-lag settings from each scenario's EddyPro files so the
# differences between runs are auditable, not hardcoded. The only setting that
# varies across runs is the time-lag treatment: the method (`tlag_meth` in the
# .eddypro) and the per-gas search window (in the .metadata).

# EddyPro time-lag method codes.
TLAG_METH = {
    "0": "None (lag removed from raw data, PWB)",
    "1": "Constant lag",
    "2": "Covariance maximization, default fallback",
    "3": "Covariance maximization, no default",
    "4": "Automatic optimization",
}


def read_tlag_meth(epro_path):
    # The tlag_meth code and label from an .eddypro file.
    m = re.search(r"^tlag_meth=(\d+)", epro_path.read_text(), re.M)
    code = m.group(1) if m else ""
    return code, TLAG_METH.get(code, "(unknown)")


def read_gas_window(meta_path, gas):
    # Nominal / min / max time lag of the first (active) `gas` column.
    text = meta_path.read_text()
    cols = {}
    for idx, key, val in re.findall(
            r"^col_(\d+)_(variable|nom_timelag|min_timelag|max_timelag)=(.*)$", text, re.M):
        cols.setdefault(idx, {})[key] = val.strip()
    # The flux is computed from the first column carrying the gas name.
    for idx in sorted(cols, key=int):
        c = cols[idx]
        if c.get("variable", "").lower() == gas:
            return (float(c["nom_timelag"]), float(c["min_timelag"]), float(c["max_timelag"]))
    return None


def window_str(code, window):
    # Human-readable lag window, phrased per time-lag method.
    if window is None:
        return "(no gas column)"
    nom, lo, hi = window
    if code == "0":
        return "removed from raw data"
    if code == "1":
        return f"constant {nom:.2f} s"
    if code == "3":  # no default fallback used
        return f"{lo:.2f} to {hi:.2f} s"
    return f"{lo:.2f} to {hi:.2f} s (nom {nom:.2f})"  # methods 2, 4 use nominal


sdir = DATA / "00-eddypro_settings"
rows = []
for cscode in sorted({scenario_of(f.name) for f in sdir.glob("*.eddypro")}):
    epro = next(iter(sorted(sdir.glob(f"{cscode}_*.eddypro"))), None)
    meta = next(iter(sorted(sdir.glob(f"{cscode}_*.metadata"))), None)
    code, label = read_tlag_meth(epro) if epro else ("", "(missing)")
    n2o_w = read_gas_window(meta, "n2o") if meta else None
    ch4_w = read_gas_window(meta, "ch4") if meta else None
    rows.append({
        "scenario": cscode,
        "tlag method": f"{code}: {label}",
        "N2O lag": window_str(code, n2o_w),
        "CH4 lag": window_str(code, ch4_w),
        ".eddypro": epro.name if epro else "(missing)",
    })
pd.DataFrame(rows)

,scenario,tlag method,N2O lag,CH4 lag,.eddypro
0,LGR-1,"3: Covariance maximization, no default",-0.05 to 10.00 s,-0.05 to 10.00 s,LGR-1_OPENLAG-10s_2021_2.eddypro
1,LGR-2,"2: Covariance maximization, default fallback",0.00 to 10.00 s (nom 1.75),0.00 to 10.00 s (nom 1.75),LGR-2_DEFAULT-10s_2021_2.eddypro
2,LGR-3,"2: Covariance maximization, default fallback",1.50 to 3.30 s (nom 1.75),1.50 to 3.30 s (nom 1.75),LGR-3_DEFAULT-NARROW_2021_2.eddypro
3,LGR-4,1: Constant lag,constant 1.75 s,constant 1.75 s,LGR-4_CONSTANT_2021_2.eddypro
4,LGR-5,"0: None (lag removed from raw data, PWB)",removed from raw data,removed from raw data,LGR-5_PWB-LAG-REMOVED_2021_2.eddypro
5,QCL-1,"3: Covariance maximization, no default",-0.05 to 10.00 s,-0.05 to 10.00 s,QCL-1_OPENLAG-10s_2021_1.eddypro
6,QCL-2,"2: Covariance maximization, default fallback",0.00 to 10.00 s (nom 0.60),0.00 to 10.00 s (nom 0.65),QCL-2_DEFAULT-10s_2021_1.eddypro
7,QCL-3,"2: Covariance maximization, default fallback",0.40 to 0.90 s (nom 0.60),0.45 to 0.90 s (nom 0.65),QCL-3_DEFAULT-NARROW_2021_1.eddypro
8,QCL-4,1: Constant lag,constant 0.60 s,constant 0.65 s,QCL-4_CONSTANT_2021_1.eddypro
9,QCL-5,"0: None (lag removed from raw data, PWB)",removed from raw data,removed from raw data,QCL-5_PWB-LAG-REMOVED_2021_1.eddypro


## PWB time-lag summaries (`00-pwb_tlag_summary/`)

The PWB detect-and-remove output for the `*-5` scenario: one
`*_detect_and_remove_tlag_summary.csv` per analyzer (with a `*_columns.md`
describing its columns). These hold the per-chunk time-lag results only; the PWB
fluxes themselves live with the other scenarios in `00-eddypro_fluxes_level-1/`.

**Table.** PWB detect-and-remove output in `data/00-pwb_tlag_summary/`: the
per-chunk time-lag summary of the `*-5` version, one file per analyzer.

In [4]:
folder_table(DATA / "00-pwb_tlag_summary")

,file,scenario,size
0,LGR-5_detect_and_remove_tlag_summary.csv,LGR-5,4.0 MB
1,LGR-5_detect_and_remove_tlag_summary_columns.md,LGR-5,5.5 KB
2,QCL-5_detect_and_remove_tlag_summary.csv,QCL-5,4.8 MB
3,QCL-5_detect_and_remove_tlag_summary_columns.md,QCL-5,5.5 KB


## Meteo (`00-meteo/`)

Supporting meteorological data. Empty for now.

In [5]:
meteo = folder_table(DATA / "00-meteo")
if len(meteo):
    # Caption above the table, matching the convention used across the site.
    display(Markdown("**Table:** supporting meteorological files in `data/00-meteo/`."))
    display(meteo)
else:
    display(Markdown("_Empty: meteo data not added yet._"))

_Empty: meteo data not added yet._

## Derived stages (tracked Parquet)

Regenerated by the pipeline notebooks from the `00-*` inputs, and committed
as a snapshot of every stage.

In [6]:
DERIVED = [
    "01-eddypro_fluxes_level-1_parquet",
    "01-pwb_tlag_summary_parquet",
    "05-flux_processing_chain_parquet",
    "04-flux-product-2025.3_subset_2024",
    "05-merged_variants_fluxproduct",
]
for sub in DERIVED:
    # Caption above each table, matching the convention used across the site.
    display(Markdown(f"**Table:** tracked Parquet files in `data/{sub}/`."))
    display(folder_table(DATA / sub, "*.parquet"))

**`01-eddypro_fluxes_level-1_parquet/`**

,file,scenario,size
0,LGR-1.parquet,LGR-1,13.3 MB
1,LGR-2.parquet,LGR-2,13.3 MB
2,LGR-3.parquet,LGR-3,13.3 MB
3,LGR-4.parquet,LGR-4,13.3 MB
4,LGR-5.parquet,LGR-5,13.3 MB
5,QCL-1.parquet,QCL-1,16.4 MB
6,QCL-2.parquet,QCL-2,16.4 MB
7,QCL-3.parquet,QCL-3,16.4 MB
8,QCL-4.parquet,QCL-4,16.4 MB
9,QCL-5.parquet,QCL-5,16.4 MB


**`01-pwb_tlag_summary_parquet/`**

,file,scenario,size
0,LGR-5_pwb_tlag.parquet,LGR-5,984.6 KB
1,QCL-5_pwb_tlag.parquet,QCL-5,1.2 MB


**`05-flux_processing_chain_parquet/`**

,file,scenario,size
0,LGR-1_FCH4_fpc.parquet,LGR-1,597.6 KB
1,LGR-1_FN2O_fpc.parquet,LGR-1,722.6 KB
2,LGR-2_FCH4_fpc.parquet,LGR-2,586.6 KB
3,LGR-2_FN2O_fpc.parquet,LGR-2,722.1 KB
4,LGR-3_FCH4_fpc.parquet,LGR-3,566.3 KB
5,LGR-3_FN2O_fpc.parquet,LGR-3,711.0 KB
6,LGR-4_FCH4_fpc.parquet,LGR-4,560.6 KB
7,LGR-4_FN2O_fpc.parquet,LGR-4,685.6 KB
8,LGR-5_FCH4_fpc.parquet,LGR-5,573.0 KB
9,LGR-5_FN2O_fpc.parquet,LGR-5,684.6 KB


**`04-flux-product-2025.3_subset_2024/`**

,file,scenario,size
0,CH-CHA_FP2025.3_subset_2024.parquet,,4.4 MB


**`05-merged_variants_fluxproduct/`**

,file,scenario,size
0,merged_variants_fp2025.3_2024.parquet,,6.0 MB


## Figures (`figures/`)

The exported figures, listed from disk so that new files appear automatically.
They fall into two groups: the Level-1 (pre-quality-control) figures from
notebooks 03, 04 and the supplement, and the quality-controlled figures from
notebooks 08 and 09. The figures themselves, with their captions, are shown in
the [Figure gallery](../docs/figure-gallery.md).


**Table.** Exported figures in `figures/`, one row per file with its size. The
file name states the notebook that wrote it (`03_`, `04_`, `08_`, `09_`, and
`suppl_` for the supplement) and the gas it shows.


In [ ]:
fig_files = folder_table(FIGDIR, "*.png")
display(fig_files)
print(f"{len(fig_files)} figure files tracked in {FIGDIR}")


## Runtime

In [8]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")

Start:    2026-07-14 01:47:18
End:      2026-07-14 01:47:20
Runtime:  0:00:02.290808
